In [1]:
import pandas as pd
import numpy as np
import random

from faker import Faker

fake = Faker("en_IN")

random.seed(42)
np.random.seed(42)
Faker.seed(42)

In [2]:
customers_df = pd.read_csv("Customers1.csv")

accounts_df = pd.read_csv("Accounts.csv")

In [3]:
customers_df["DOB"] = pd.to_datetime(customers_df["DOB"])

today = pd.Timestamp.today()

customers_df["Age"] = (
    (today - customers_df["DOB"]).dt.days // 365
)

In [4]:
active_customers = accounts_df[
    accounts_df["Status"] == "Active"
]["Customer_ID"].unique()

customers_df = customers_df[
    customers_df["Customer_ID"].isin(active_customers)
].copy()

In [5]:
eligible_fd = customers_df[

    (customers_df["Age"] >= 30) &
    (customers_df["Annual_Income"] >= 500000)

].copy()

In [6]:
eligible_fd.shape

(5495, 15)

In [7]:
fd_customers = eligible_fd.sample(
    n=2000,
    random_state=42
).reset_index(drop=True)

In [8]:
fd_customers.shape

(2000, 15)

In [9]:
FD_DURATIONS = [

    6,
    12,
    24,
    36,
    60,
    84,
    120

]

FD_DURATION_WEIGHTS = [

    10,
    30,
    20,
    15,
    15,
    5,
    5

]

INTEREST_RATE = {

    6: 5.50,

    12: 6.50,

    24: 6.80,

    36: 7.00,

    60: 7.20,

    84: 7.30,

    120: 7.40

}

FD_STATUS = [

    "Active",
    "Matured",
    "Closed"

]

FD_STATUS_WEIGHTS = [

    65,
    20,
    15

]

In [10]:
fixed_deposits = []

fd_number = 1

In [11]:
for _, customer in fd_customers.iterrows():

    customer_id = customer["Customer_ID"]

    income = customer["Annual_Income"]

    join_date = pd.to_datetime(customer["Join_Date"])

    # Deposit Amount
    lower = 50000

    upper = min(int(income * 2), 5000000)

    deposit_amount = random.randint(
        lower,
        upper
    )

    # Duration
    duration = random.choices(
        FD_DURATIONS,
        weights=FD_DURATION_WEIGHTS,
        k=1
    )[0]

    # Interest Rate
    interest_rate = INTEREST_RATE[duration]

    # Start Date
    start_date = fake.date_between(
        start_date=join_date.date(),
        end_date="today"
    )

    # Maturity Date
    maturity_date = (
        pd.to_datetime(start_date) +
        pd.DateOffset(months=duration)
    )

    # Simple annual compounding
    years = duration / 12

    maturity_amount = round(
        deposit_amount *
        ((1 + interest_rate / 100) ** years),
        2
    )

    status = random.choices(
        FD_STATUS,
        weights=FD_STATUS_WEIGHTS,
        k=1
    )[0]

    fd_id = f"FD{fd_number:06d}"

    fd_number += 1

    fixed_deposits.append({

        "FD_ID": fd_id,

        "Customer_ID": customer_id,

        "Deposit_Amount": deposit_amount,

        "Interest_Rate": interest_rate,

        "Duration_Months": duration,

        "Maturity_Amount": maturity_amount,

        "Start_Date": start_date,

        "Maturity_Date": maturity_date,

        "Status": status

    })

In [12]:
fixed_deposits_df = pd.DataFrame(fixed_deposits)

fixed_deposits_df.shape

(2000, 9)

In [13]:
fixed_deposits_df.head()

,FD_ID,Customer_ID,Deposit_Amount,Interest_Rate,Duration_Months,Maturity_Amount,Start_Date,Maturity_Date,Status
0,FD000001,C03427,283478,5.5,6,291169.31,2024-08-29,2025-02-28,Active
1,FD000002,C01293,284053,6.5,12,302516.45,2020-09-08,2021-09-08,Active
2,FD000003,C03426,1603292,7.2,60,2269794.57,2021-02-27,2026-02-27,Active
3,FD000004,C02047,934834,5.5,6,960197.85,2025-09-11,2026-03-11,Active
4,FD000005,C09074,2001701,6.8,24,2283188.20,2022-08-26,2024-08-26,Active


In [14]:
fixed_deposits_df.isnull().sum()

FD_ID              0
Customer_ID        0
Deposit_Amount     0
Interest_Rate      0
Duration_Months    0
Maturity_Amount    0
Start_Date         0
Maturity_Date      0
Status             0
dtype: int64

In [15]:
fixed_deposits_df["FD_ID"].duplicated().sum()

np.int64(0)

In [16]:
fixed_deposits_df["Status"].value_counts(normalize=True) * 100

Status
Active     64.60
Matured    20.85
Closed     14.55
Name: proportion, dtype: float64

In [17]:
fixed_deposits_df["Duration_Months"].value_counts().sort_index()

Duration_Months
6      208
12     647
24     404
36     288
60     272
84      86
120     95
Name: count, dtype: int64

In [18]:
(
    fixed_deposits_df["Maturity_Amount"] >=
    fixed_deposits_df["Deposit_Amount"]
).all()

np.True_

In [19]:
fixed_deposits_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   FD_ID            2000 non-null   object        
 1   Customer_ID      2000 non-null   object        
 2   Deposit_Amount   2000 non-null   int64         
 3   Interest_Rate    2000 non-null   float64       
 4   Duration_Months  2000 non-null   int64         
 5   Maturity_Amount  2000 non-null   float64       
 6   Start_Date       2000 non-null   object        
 7   Maturity_Date    2000 non-null   datetime64[ns]
 8   Status           2000 non-null   object        
dtypes: datetime64[ns](1), float64(2), int64(2), object(4)
memory usage: 140.8+ KB


In [20]:
fixed_deposits_df["Start_Date"] = pd.to_datetime(fixed_deposits_df["Start_Date"])

In [21]:
fixed_deposits_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   FD_ID            2000 non-null   object        
 1   Customer_ID      2000 non-null   object        
 2   Deposit_Amount   2000 non-null   int64         
 3   Interest_Rate    2000 non-null   float64       
 4   Duration_Months  2000 non-null   int64         
 5   Maturity_Amount  2000 non-null   float64       
 6   Start_Date       2000 non-null   datetime64[ns]
 7   Maturity_Date    2000 non-null   datetime64[ns]
 8   Status           2000 non-null   object        
dtypes: datetime64[ns](2), float64(2), int64(2), object(3)
memory usage: 140.8+ KB


In [22]:
fixed_deposits_df.to_csv(
    "Fixed_Deposits.csv",
    index=False
)